# Ollama local LLM setup

Поднимает небольшую instruct-модель через **Ollama** и проверяет OpenAI-compatible API (`/v1/chat/completions`).

**Установка Ollama:** https://ollama.com/download

```bash
ollama serve
ollama pull qwen2.5:1.5b
```

Альтернативы: `phi3:mini`, `llama3.2:1b`, `qwen2.5:3b`.

In [1]:
import os

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")
MODEL_ID = os.environ.get("OLLAMA_MODEL", "qwen2.5:1.5b")
BASE_URL = f"{OLLAMA_HOST.rstrip('/')}/v1"

print("Host:", OLLAMA_HOST)
print("Model:", MODEL_ID)
print("OpenAI base:", BASE_URL)

Host: http://127.0.0.1:11434
Model: qwen2.5:1.5b
OpenAI base: http://127.0.0.1:11434/v1


In [3]:
# Health check
import urllib.request, json

with urllib.request.urlopen(f"{OLLAMA_HOST.rstrip('/')}/api/tags", timeout=5) as resp:
    data = json.load(resp)

names = [m.get("name") for m in data.get("models", [])]
print("Installed models:", names)
assert any(MODEL_ID in (n or "") for n in names), f"Pull the model first: ollama pull {MODEL_ID}"

Installed models: ['qllama/multilingual-e5-large:latest', 'thirdeyeai/DeepSeek-R1-Distill-Qwen-7B-uncensored:latest', 'ai_elcid/pygmalion2-13b:latest', 'mxbai-embed-large:latest', 'nomic-embed-text:latest', 'owl/t-lite:latest', 'llama2-uncensored:latest']


AssertionError: Pull the model first: ollama pull qwen2.5:1.5b

In [ ]:
# OpenAI-compatible smoke test
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key="ollama")
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "Отвечай кратко на русском."},
        {"role": "user", "content": "Скажи одним предложением, что такое RAG."},
    ],
    max_tokens=128,
    temperature=0.2,
)
print(resp.choices[0].message.content)


```env
OLLAMA_BASE_URL=http://127.0.0.1:11434/v1
OLLAMA_API_KEY=ollama
OLLAMA_MODEL=qwen2.5:1.5b
```

Затем:

```bash
/usr/bin/python3 scripts/evaluate_zeroshot.py --backend ollama
/usr/bin/python3 scripts/evaluate_rag.py --backend ollama
```